In [ ]:
# CORDEX Future Extreme Rainfall Frequency Analysis

This notebook quantifies projected changes in the **frequency of extreme rainfall events** using daily CORDEX climate model output.
    
Extreme events are defined using the **same fixed daily precipitation threshold** applied in the observed PRISM analysis, enabling

conceptual continuity between historical observations and future projections.The analysis compares two future time windows to identify
    
**directional changes in event frequency**, without assuming stationarity or fitting probabilistic return-period models.

In [ ]:
## Purpose

- Load daily CORDEX precipitation data (NetCDF)
- Subset to a region of interest
- Identify extreme rainfall events using a fixed threshold
- Count exceedances across two future epochs
- Calculate the difference in event frequency between epochs

This notebook focuses on **frequency change only**.

Flood probability interpretation and exposure analysis occur in downstream notebooks.


In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature


In [ ]:
# -----------------------------
# USER CONFIGURATION
# -----------------------------

# Path to CORDEX NetCDF file
CORDEX_PATH = Path("../data/raw/cordex_precip_daily.nc")

# Variable name in NetCDF (matches thesis)
PRECIP_VAR = "prec"

# Extreme rainfall threshold (mm/day)
# 3 inches = 76.2 mm
EXTREME_THRESHOLD_MM = 76.2

# Spatial subset (North Carolina region from thesis)
LAT_SLICE = slice(33.8, 37.0)
LON_SLICE = slice(-84.7, -75.4)

# Future epochs (matches thesis logic)
EARLY_START = "2021-01-01"
EARLY_END   = "2050-01-01"

LATE_START  = "2051-01-01"
LATE_END    = "2080-01-01"


In [ ]:
if not CORDEX_PATH.exists():
    raise FileNotFoundError(
        f"CORDEX file not found at {CORDEX_PATH}. "
        "Place NetCDF file in data/raw/ before proceeding."
    )


In [ ]:
ds = xr.open_dataset(CORDEX_PATH)

precip = ds[PRECIP_VAR].sel(
    lat=LAT_SLICE,
    lon=LON_SLICE
)

precip


In [ ]:
# Boolean exceedance array
extreme_events = precip >= EXTREME_THRESHOLD_MM


In [ ]:
early_counts = extreme_events.sel(
    time=slice(EARLY_START, EARLY_END)
).sum(dim="time")

late_counts = extreme_events.sel(
    time=slice(LATE_START, LATE_END)
).sum(dim="time")


In [ ]:
frequency_diff = late_counts - early_counts


In [ ]:
fig = plt.figure(figsize=(16, 9), dpi=150)
ax = plt.axes(projection=ccrs.Mercator())

ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.5)
ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
ax.add_feature(cfeature.STATES.with_scale("50m"), lw=0.3)

gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5)
gl.xlabel_style = {"size": 7}
gl.ylabel_style = {"size": 7}

frequency_diff.plot.contourf(
    ax=ax,
    transform=ccrs.PlateCarree(),
    levels=21,
    cbar_kwargs={
        "orientation": "horizontal",
        "shrink": 0.6,
        "pad": 0.05,
        "label": "Change in extreme rainfall event frequency"
    }
)

ax.set_extent([-90, -75, 33, 40], crs=ccrs.PlateCarree())
plt.title("Projected Change in Extreme Rainfall Frequency (CORDEX)")
plt.show()


In [ ]:
OUTPUT_TABLE = Path("../outputs/tables/cordex_future_frequency_summary.csv")
OUTPUT_TABLE.parent.mkdir(parents=True, exist_ok=True)

summary_df = pd.DataFrame({
    "early_epoch_total": [early_counts.sum().item()],
    "late_epoch_total": [late_counts.sum().item()],
    "difference": [frequency_diff.sum().item()]
})

summary_df.to_csv(OUTPUT_TABLE, index=False)

print(f"Saved CORDEX frequency summary to {OUTPUT_TABLE}")


In [ ]:
OUTPUT_MAP = Path("../outputs/maps/cordex_frequency_change.png")
OUTPUT_MAP.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(OUTPUT_MAP, bbox_inches="tight", dpi=150)
print(f"Saved map to {OUTPUT_MAP}")


In [ ]:
## Next Steps

This notebook produces projected changes in extreme rainfall **event frequency** using CORDEX climate model output.

These results are used in:

- `05_flood_probability_context.ipynb`  
  → to interpret how frequency changes may affect flood behavior and recovery time

Users may proceed directly to the next notebook after confirming this step completes successfully.
